# Prerequisites
1. Need to have a python environment, we recommend Conda: Install Conda from https://www.anaconda.com/download/success?reg=skipped 
2. Need to have COPT installed: Fill out the form from https://copt.shanshu.ai/copt and wait for an email for further instructions for installation

To create the environment:

In [ ]:
conda env create --name <name> --file environment.yml

To activate the environment:

In [ ]:
conda activate <name>

If you get an error make sure all libraries are installed inside of your environment.

Specifically check:
1. coptpy
2. networkx
3. natsort

To install a library in the environment make sure the environment is active then:

In [ ]:
pip install <name>

These are the imports used to run COPT as well as read the graphs.

In [ ]:
# Imports
import os
import numpy as np
import coptpy as cp
import coptpy as cp
import networkx as nx
from coptpy import COPT
from natsort import natsorted

read_nxgraph reads the input file and constructs the graph based on the contents of the file.

In [ ]:
def read_nxgraph(filename: str) -> nx.Graph(): # type: ignore
    graph = nx.Graph()
    with open(filename, 'r') as file:
        line = file.readline()
        is_first_line = True
        while line is not None and line != '':
            if '//' not in line:
                if is_first_line:
                    strings = line.split(" ")
                    num_nodes = int(strings[0])
                    num_edges = int(strings[1])
                    nodes = list(range(num_nodes))
                    graph.add_nodes_from(nodes)
                    is_first_line = False
                else:
                    node1, node2, weight = line.split()
                    graph.add_edge(int(node1), int(node2), weight=weight)
            line = file.readline()
    return graph

copt_maxcut constructs the COPT solver and utilizes it to solve a given graph.

In [ ]:
def copt_maxcut(graph, env):
    # Create QUBO matrix
    nodes = len(list(graph.nodes))
    J = nx.to_numpy_array(graph)
    
    # Construct model
    model = env.createModel("qubo")

    # Set time limit 1 hr
    model.setLogFile("../logs/copt.log")
    model.setParam(COPT.Param.TimeLimit, 3600.0)
    model.setParam(COPT.Param.RelGap, 0.0)

    # Create variable for each vertex
    x = [ model.addVar(vtype=COPT.BINARY, name=f"x_{i}") for i in range(nodes) ]
                
    # Linearize x_i * x_j to y_ij
    y = np.zeros((nodes, nodes), dtype=object)
    for i in range(nodes):
        for j in range(i +1, nodes):
            y[i, j] = model.addVar(vtype=COPT.BINARY, name=f"y_{i},{j}")
            model.addConstr(y[i, j] <= x[i])
            model.addConstr(y[i, j] <= x[j])
            model.addConstr(y[i, j] >= x[i] + x[j] - 1)

    # Objective function
    model.setObjective(cp.quicksum(
        -J[i, j] * ( (4 * y[i,j]) + (-2 * x[i]) + (-2 * x[j]) + 1 )
        for i in range(nodes) for j in range(i + 1, nodes) if J[i, j] != 0.0 
    ), sense=COPT.MINIMIZE)

    # Solve
    model.solve()
    sol = ""
    for i in range(nodes):
        sol += f"{int(x[i].x)}"
    
    return model.BestObj, sol

Test code to run the solver.

In [ ]:
# Parse all files in data_dir
data_dir = "../tutorials/VCA/dataset/EA/EA_40x40"
input_files = natsorted(os.listdir(data_dir))

# Write directories & files
results_dir = f"../results/{'/'.join(data_dir.split('/')[2:])}"
solutions_dir = f"../solutions/{'/'.join(data_dir.split('/')[2:])}"

os.makedirs(results_dir, exist_ok=True)
os.makedirs(solutions_dir, exist_ok=True)
os.makedirs("../logs", exist_ok=True)

# Create COPT environment
env = cp.Envr()

for file in input_files:
    graph = read_nxgraph(f"{data_dir}/{file}")
    obj_val, sol = copt_maxcut(graph, env)

    with open(f"{results_dir}/COPT.txt", "a") as f:
        f.write(f"{obj_val}\n")
    with open(f"{solutions_dir}/COPT.txt", "a") as f:
        f.write(f"{sol}\n")